# Week 1 — LLM Fundamentals & Environment Setup

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-01-llm-fundamentals-content.html`. Run the cells in order during lab time.

**You will practice:**
1. Making your first Gemini API call from Python.
2. Seeing `temperature`, `top_p`, and `top_k` change model output on the *same* prompt.
3. Counting tokens for real and comparing it to the "1 token ≈ 4 characters" rule of thumb.
4. Making the *same* call through three tools: the raw SDK, a minimal Google ADK agent, and a minimal LangChain chat model.
5. Two open exercises.

**Before you start:** create a free API key at [Google AI Studio](https://aistudio.google.com/app/apikey) and save it in a `.env` file next to this notebook:

```
GOOGLE_API_KEY="paste-your-key-here"
```


In [ ]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv

## 1. Environment check

Load the API key and make one call to confirm everything is wired up.

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in a .env file first."

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"

response = client.models.generate_content(
    model=MODEL,
    contents="In one sentence, what is a large language model?",
)
print(response.text)

## 2. Sampling parameters: temperature, top-p, top-k

We'll send the **same prompt** several times, changing only `temperature`. Watch how the output goes from
predictable to varied.

In [ ]:
PROMPT = "Give me one creative name for a coffee shop. Reply with just the name."

for temp in [0.0, 0.4, 0.9, 1.5]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=temp, max_output_tokens=20),
    )
    print(f"temperature={temp:<4} -> {response.text.strip()}")

Run the cell above a few times. At `temperature=0.0` the answer should barely change between runs.
At `temperature=1.5` you should see real variety (and occasionally something a little unhinged — that's expected).

Now let's isolate `top_p` and `top_k` by holding temperature fixed at a mid-range value.

In [ ]:
for top_p in [0.1, 0.5, 0.95]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=0.9, top_p=top_p, max_output_tokens=20),
    )
    print(f"top_p={top_p:<5} -> {response.text.strip()}")

print()
for top_k in [1, 5, 40]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=0.9, top_k=top_k, max_output_tokens=20),
    )
    print(f"top_k={top_k:<3} -> {response.text.strip()}")

## 3. Tokens: count them for real

The rule of thumb is "1 token ≈ 4 characters ≈ 0.75 words" for English. Let's check it against the real
tokenizer using `count_tokens`.

In [ ]:
samples = [
    "Hi!",
    "The capital of France is Paris.",
    "AI Agentic Engineering is a course about building autonomous systems with large language models, "
    "retrieval-augmented generation, and multi-agent orchestration frameworks like Google ADK and LangGraph.",
]

for text in samples:
    result = client.models.count_tokens(model=MODEL, contents=text)
    chars = len(text)
    tokens = result.total_tokens
    ratio = chars / tokens if tokens else 0
    print(f"chars={chars:<4} tokens={tokens:<4} chars/token={ratio:.2f}  | {text[:50]!r}")

## 4. Same call, three tools

Below, the identical prompt is sent through: **(A)** the raw `google-genai` SDK, **(B)** a one-line **Google ADK**
agent run with `InMemoryRunner`, and **(C)** a **LangChain** chat model. This is a *light preview* — we are not
building real agents yet (tools, multi-step reasoning, ReAct) until Corte 2, Week 6 onward. The goal today is
just to recognize the same underlying API call under each interface.

In [ ]:
# --- A. Raw SDK ---
prompt = "Name one famous mathematician and the theorem they're best known for."

response = client.models.generate_content(model=MODEL, contents=prompt)
print("A) raw SDK:\n", response.text)

In [ ]:
# --- B. Google ADK (minimal agent) ---
import asyncio
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

adk_agent = Agent(
    model=MODEL,
    name="week1_demo_agent",
    instruction="Answer concisely, in 1-2 sentences.",
)

def ask_adk_agent(agent, prompt, app_name="week1_app", user_id="student"):
    """Small reusable helper: runs one prompt through an ADK agent and returns the final text."""
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = asyncio.run(runner.session_service.create_session(app_name=app_name, user_id=user_id))
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = None
    for event in runner.run(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts and event.content.parts[0].text:
            final_text = event.content.parts[0].text
    return final_text

print("B) Google ADK agent:\n", ask_adk_agent(adk_agent, prompt))

In [ ]:
# --- C. LangChain chat model ---
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=MODEL)
lc_response = llm.invoke(prompt)
print("C) LangChain:\n", lc_response.content)

## 5. Exercises

Complete both before the peer code review activity.

In [ ]:
# TODO Exercise 1 — Your own parameter sweep
# Pick a prompt of your own (something with a clearly "creative" vs "factual" framing).
# Sweep temperature over at least 4 values and print the results, like section 2 above.

MY_PROMPT = "..."  # replace this

# your sweep here


In [ ]:
# TODO Exercise 2 — Estimate tokens without calling the API
# Write a function `estimate_tokens(text)` that approximates token count using ONLY the
# "1 token ≈ 4 characters" rule (no API call). Then compare it against the real count_tokens()
# result for the three `samples` from section 3, and print the % error for each.

def estimate_tokens(text: str) -> float:
    # your implementation here
    ...

for text in samples:
    real = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    estimate = estimate_tokens(text)
    # print a comparison line with % error here


## Next week

Week 2 — **Context Engineering I**: system prompts, few-shot prompting, chain-of-thought, and forcing structured
JSON output. See `week-02-context-engineering-i-content.html`.